In [51]:
# ==========================================================
# SMART Survey Processing Pipeline
# BIT29B - Part 1
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import re
from collections import Counter

In [52]:
# ----------------------------------------------------------
# Project folders
# ----------------------------------------------------------

PROJECT_DIR = Path(r"C:\Users\abdinasir\Desktop\group2 bit29b")

SURVEY_DIR = PROJECT_DIR / "Smart Survey Data Sets"

OUTPUT_DIR = PROJECT_DIR / "organized_surveys"

OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("SMART Survey Processing Pipeline")
print("=" * 60)
print(f"Project Folder : {PROJECT_DIR}")
print(f"Survey Folder  : {SURVEY_DIR}")
print(f"Output Folder  : {OUTPUT_DIR}")

SMART Survey Processing Pipeline
Project Folder : C:\Users\abdinasir\Desktop\group2 bit29b
Survey Folder  : C:\Users\abdinasir\Desktop\group2 bit29b\Smart Survey Data Sets
Output Folder  : C:\Users\abdinasir\Desktop\group2 bit29b\organized_surveys


In [33]:
# ----------------------------------------------------------
# Locate all SMART survey files
# ----------------------------------------------------------

survey_files = sorted(SURVEY_DIR.rglob("*.as"))

print("=" * 60)
print("SMART Survey Files")
print("=" * 60)

print(f"Total .as files found : {len(survey_files)}")

print("\nFirst five files:")

for file in survey_files[:5]:
    print(file.name)

SMART Survey Files
Total .as files found : 80

First five files:
Adado_SMART_survey_2022_final.as
Burco Urban_IDPs Complete Anthro_Mortality.as
Dhobley ENA Anthro & Mortality.as
Elbarde_Som_Jan_2023.as
Hudur_Som_Jan_2023.as


In [34]:
# ----------------------------------------------------------
# Build survey inventory
# ----------------------------------------------------------

survey_inventory = pd.DataFrame({
    "File Name": [f.name for f in survey_files],
    "Folder": [f.parent.name for f in survey_files],
    "Extension": [f.suffix.lower() for f in survey_files]
})

print("=" * 60)
print("Survey Inventory")
print("=" * 60)

print(f"Total survey files : {len(survey_inventory)}")

display(survey_inventory.head())

Survey Inventory
Total survey files : 80


,File Name,Folder,Extension
0,Adado_SMART_survey_2022_final.as,Smart Survey Data Sets,.as
1,Burco Urban_IDPs Complete Anthro_Mortality.as,Smart Survey Data Sets,.as
2,Dhobley ENA Anthro & Mortality.as,Smart Survey Data Sets,.as
3,Elbarde_Som_Jan_2023.as,Smart Survey Data Sets,.as
4,Hudur_Som_Jan_2023.as,Smart Survey Data Sets,.as


In [35]:
# ----------------------------------------------------------
# Read survey headers
# ----------------------------------------------------------

def read_header(file_path, max_lines=120):
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
            return "".join([next(file) for _ in range(max_lines)])
    except StopIteration:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
            return file.read()
    except Exception:
        return ""

survey_inventory["Header"] = survey_inventory["File Name"].apply(
    lambda name: read_header(SURVEY_DIR / name)
)

print("=" * 60)
print("Survey headers loaded successfully.")
print("=" * 60)

print(f"Headers extracted : {len(survey_inventory)}")

Survey headers loaded successfully.
Headers extracted : 80


In [36]:
# ============================================================
# CELL 6 - BUILD SURVEY METADATA
# ============================================================

import re
import pandas as pd

# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def split_cells(line):
    return line.rstrip("\n").split("\t")


def has_aggregate_data(lines):
    """Return True if aggregate mortality data exists."""

    in_section = False

    for line in lines:

        text = line.lower()

        if "mortality_new" in text:
            in_section = True
            continue

        if in_section and not line.strip():
            break

        if in_section:

            cells = split_cells(line)

            if cells and re.fullmatch(r"\d+", cells[0].strip()):
                return True

    return False


def has_individual_data(lines):
    """Return True if individual mortality data exists."""

    in_section = False
    header_found = False

    for line in lines:

        text = line.lower()

        if "mor_individual" in text or "mortality_individual" in text:
            in_section = True
            header_found = False
            continue

        if in_section and not line.strip():
            break

        if in_section:

            if (
                not header_found
                and "p1_sex" in text
                and "p1_age" in text
            ):
                header_found = True
                continue

            if header_found:

                cells = split_cells(line)

                if len(cells) > 3:

                    if any(c.strip() for c in cells[1:]):
                        return True

    return False


def has_anthropometry(lines):
    """Check for anthropometry table."""

    for line in lines:

        text = line.lower()

        if (
            "weight" in text
            and "height" in text
            and "muac" in text
        ):
            return True

    return False


def detect_survey_level(filename):
    """
    Survey Level Rule

    LHZ in filename -> LHZ
    Otherwise       -> Admin2
    """

    if "lhz" in filename.lower():
        return "LHZ"

    return "Admin2"


# ------------------------------------------------------------
# Build Metadata
# ------------------------------------------------------------

metadata = []

print("=" * 70)
print("Building Survey Metadata")
print("=" * 70)

for survey in survey_files:

    with open(
        survey,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        lines = f.readlines()

    has_agg = has_aggregate_data(lines)
    has_ind = has_individual_data(lines)

    if has_agg:
        survey_type = "Aggregate"

    elif has_ind:
        survey_type = "Individual"

    else:
        survey_type = "Issue"

    survey_level = detect_survey_level(survey.name)

    destination_folder = (
        f"{survey_level.lower()}/{survey_type.lower()}"
    )

    metadata.append({

        "file_name": survey.name,

        "survey_type": survey_type,

        "survey_level": survey_level,

        "has_mortality": has_agg or has_ind,

        "has_anthropometry": has_anthropometry(lines),

        "destination_folder": destination_folder

    })

# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

survey_metadata = pd.DataFrame(metadata)

# ------------------------------------------------------------
# Save Metadata
# ------------------------------------------------------------

final_folder = PROJECT_DIR / "data" / "final"
final_folder.mkdir(parents=True, exist_ok=True)

metadata_file = final_folder / "survey_metadata.csv"

survey_metadata.to_csv(
    metadata_file,
    index=False
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nSurvey Type Summary")
print("-" * 40)
print(survey_metadata["survey_type"].value_counts())

print("\nSurvey Level Summary")
print("-" * 40)
print(survey_metadata["survey_level"].value_counts())

print("\nMetadata saved to:")
print(metadata_file)

print("\nFirst 10 Surveys")

display(survey_metadata.head(10))

Building Survey Metadata

Survey Type Summary
----------------------------------------
survey_type
Individual    65
Aggregate     14
Issue          1
Name: count, dtype: int64

Survey Level Summary
----------------------------------------
survey_level
Admin2    61
LHZ       19
Name: count, dtype: int64

Metadata saved to:
C:\Users\abdinasir\Desktop\group2 bit29b\data\final\survey_metadata.csv

First 10 Surveys


,file_name,survey_type,survey_level,has_mortality,has_anthropometry,destination_folder
0,Adado_SMART_survey_2022_final.as,Individual,Admin2,True,True,admin2/individual
1,Burco Urban_IDPs Complete Anthro_Mortality.as,Individual,Admin2,True,True,admin2/individual
2,Dhobley ENA Anthro & Mortality.as,Individual,Admin2,True,True,admin2/individual
3,Elbarde_Som_Jan_2023.as,Individual,Admin2,True,True,admin2/individual
4,Hudur_Som_Jan_2023.as,Individual,Admin2,True,True,admin2/individual
5,Odweyne_rural Complete Anthro_Mortality.as,Individual,Admin2,True,True,admin2/individual
6,SOM_2013_admin2_Baydhaba.as,Aggregate,Admin2,True,True,admin2/aggregate
7,SOM_2013_admin2_BeletWeyne.as,Aggregate,Admin2,True,True,admin2/aggregate
8,SOM_2013_admin2_BeletWeyne_2.as,Aggregate,Admin2,True,True,admin2/aggregate
9,SOM_2013_admin2_Mogadishu_IDP.as,Aggregate,Admin2,True,True,admin2/aggregate


In [37]:
# ============================================================
# CELL 7 - ORGANIZE SMART SURVEY FILES
# ============================================================

import shutil

print("=" * 70)
print("Organizing SMART Survey Files")
print("=" * 70)

# ------------------------------------------------------------
# Destination Root
# ------------------------------------------------------------

organized_root = PROJECT_DIR / "data" / "organized"

# Create folder structure
folders = [

    organized_root / "admin2" / "individual",
    organized_root / "admin2" / "aggregate",
    organized_root / "admin2" / "issue",

    organized_root / "lhz" / "individual",
    organized_root / "lhz" / "aggregate",
    organized_root / "lhz" / "issue"

]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Copy Surveys
# ------------------------------------------------------------

copied = 0

for _, row in survey_metadata.iterrows():

    source = SURVEY_DIR / row["file_name"]

    destination = (
        organized_root /
        row["destination_folder"] /
        row["file_name"]
    )

    if source.exists():

        shutil.copy2(source, destination)

        copied += 1

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(f"\nSurvey files copied : {copied}")

print("\nFolder Summary")
print("-" * 40)

for folder in folders:

    total = len(list(folder.glob("*.as")))

    print(f"{folder.relative_to(organized_root)} : {total}")

print("\nSurvey organization completed successfully.")

print("\nOrganized data location:")

print(organized_root)

Organizing SMART Survey Files

Survey files copied : 80

Folder Summary
----------------------------------------
admin2\individual : 53
admin2\aggregate : 7
admin2\issue : 1
lhz\individual : 12
lhz\aggregate : 7
lhz\issue : 0

Survey organization completed successfully.

Organized data location:
C:\Users\abdinasir\Desktop\group2 bit29b\data\organized


In [18]:
def build_output_lines(lines: list[str], default_month: str, default_year: str) -> list[str]:
    out: list[str] = []
    date_order = get_date_order(lines)
    surv_month, surv_year = get_month_year_from_survdate_column(lines, date_order)
    if surv_month and surv_year:
        default_month, default_year = surv_month, surv_year

    in_individual_block = False
    individual_header_seen = False
    surdate_idx = -1
    date_idx = -1
    in_aggregate_block = False
    aggregate_header_seen = False

    for line in lines:
        cells = split_cells(line)
        lower = line.lower()

        if in_aggregate_block and ("mor_individual" in lower or "mortality_individual" in lower):
            in_aggregate_block = False
            aggregate_header_seen = False

        if "mortality_new" in lower:
            in_aggregate_block = True
            aggregate_header_seen = False
            out.append(line)
            continue

        if in_aggregate_block:
            if not line.strip():
                in_aggregate_block = False
                aggregate_header_seen = False
                out.append(line)
                continue

            if not aggregate_header_seen:
                aggregate_header_seen = True
                out.append(line)
                continue

            if len(cells) >= 2:
                first_cell = cells[0].strip()
                if re.fullmatch(r"\d+", first_cell):
                    cells_copy = cells[:]
                    if len(cells_copy) >= 2:
                        cells_copy[1] = default_month
                    if len(cells_copy) >= 3:
                        cells_copy[2] = default_year
                    out.append(row_to_csv(cells_copy))
                    continue

            out.append(line)
            continue

        if "mor_individual" in lower or "mortality_individual" in lower:
            in_individual_block = True
            individual_header_seen = False
            surdate_idx = -1
            date_idx = -1
            out.append(line)
            continue

        if in_individual_block:
            if not line.strip():
                in_individual_block = False
                individual_header_seen = False
                out.append(line)
                continue

            if not individual_header_seen:
                individual_header_seen = True
                lowered_cells = [c.strip().lower() for c in cells]
                if "survdate" in lowered_cells:
                    surdate_idx = lowered_cells.index("survdate")
                if "date" in lowered_cells:
                    date_idx = lowered_cells.index("date")
                out.append(line)
                continue

            if len(cells) > max(surdate_idx, date_idx, 0):
                cells_copy = cells[:]
                
                if surdate_idx >= 0 and surdate_idx < len(cells_copy):
                    date_text = cells_copy[surdate_idx].strip()
                    if date_text:
                        month, year = parse_month_year_from_date(date_text, date_order)
                        if not month:
                            month = default_month
                        if not year:
                            year = default_year
                        cells_copy[surdate_idx] = f"{month}/{year}"

                if date_idx >= 0 and date_idx < len(cells_copy):
                    date_text = cells_copy[date_idx].strip()
                    if date_text:
                        month, year = parse_month_year_from_date(date_text, date_order)
                        if not month:
                            month = default_month
                        if not year:
                            year = default_year
                        cells_copy[date_idx] = f"{month}/{year}"

                out.append(row_to_csv(cells_copy))
                continue

            out.append(line)
            continue

        out.append(line)

    return out

In [38]:
# ============================================================
# CELL 9 - CONVERT ORGANIZED SMART SURVEYS TO CSV
# ============================================================

import pandas as pd

print("=" * 70)
print("Converting SMART Surveys")
print("=" * 70)

ORGANIZED_ROOT = PROJECT_DIR / "data" / "organized"
CSV_ROOT = PROJECT_DIR / "data" / "csv"
FINAL_ROOT = PROJECT_DIR / "data" / "final"

CSV_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_ROOT.mkdir(parents=True, exist_ok=True)

results = []

survey_files = sorted(ORGANIZED_ROOT.rglob("*.as"))

print(f"Survey files found: {len(survey_files)}")

for i, file in enumerate(survey_files, start=1):

    print(f"[{i}/{len(survey_files)}] {file.name}")

    try:

        content = file.read_text(
            encoding="utf-8",
            errors="replace"
        )

        lines = content.splitlines()

        month, year = parse_month_year(
            lines,
            file.name
        )

        output_lines = build_output_lines(
            lines,
            month,
            year
        )

        relative_folder = file.parent.relative_to(
            ORGANIZED_ROOT
        )

        output_folder = CSV_ROOT / relative_folder
        output_folder.mkdir(
            parents=True,
            exist_ok=True
        )

        output_file = output_folder / (
            file.stem + ".csv"
        )

        ok = safe_write(
            output_file,
            output_lines
        )

        if not ok:
            raise Exception("Unable to write CSV.")

        results.append({

            "file_name": file.name,
            "status": "SUCCESS",
            "rows": len(output_lines),
            "csv_file": str(
                output_file.relative_to(CSV_ROOT)
            )

        })

    except Exception as e:

        results.append({

            "file_name": file.name,
            "status": "FAILED",
            "rows": None,
            "csv_file": None,
            "error": str(e)

        })

conversion_log = pd.DataFrame(results)

log_file = FINAL_ROOT / "conversion_log.csv"

conversion_log.to_csv(
    log_file,
    index=False
)

print("\n" + "=" * 70)
print("Conversion Summary")
print("=" * 70)

print(conversion_log["status"].value_counts())

print("\nLog saved to:")
print(log_file)

display(conversion_log.head(20))

Converting SMART Surveys
Survey files found: 80
[1/80] SOM_2013_admin2_Baydhaba.as
[2/80] SOM_2013_admin2_BeletWeyne.as
[3/80] SOM_2013_admin2_BeletWeyne_2.as
[4/80] SOM_2013_admin2_Mogadishu_IDP.as
[5/80] SOM_2014_11_admin1_Bari urban.as
[6/80] SOM_2014_admin2_Burco_IDP.as
[7/80] SOM_2014_admin2_Hargeysa IDP.as
[8/80] Adado_SMART_survey_2022_final.as
[9/80] Burco Urban_IDPs Complete Anthro_Mortality.as
[10/80] Dhobley ENA Anthro & Mortality.as
[11/80] Elbarde_Som_Jan_2023.as
[12/80] Hudur_Som_Jan_2023.as
[13/80] Odweyne_rural Complete Anthro_Mortality.as
[14/80] SOM_201806_admin2_Baydhaba.as
[15/80] SOM_201806_admin2_BeletWeyne rural.as
[16/80] SOM_201806_admin2_BeletWeyne urban.as
[17/80] SOM_201806_admin2_Berbera IDP.as
[18/80] SOM_201806_admin2_Bossaso IDP.as
[19/80] SOM_201806_admin2_Burco IDP.as
[20/80] SOM_201806_admin2_Dhusamareeb IDP.as
[21/80] SOM_201806_admin2_Doolow IDP.as
[22/80] SOM_201806_admin2_Galkayo IDP.as
[23/80] SOM_201806_admin2_Garowe IDP.as
[24/80] SOM_201806_ad

,file_name,status,rows,csv_file
0,SOM_2013_admin2_Baydhaba.as,SUCCESS,901,admin2\aggregate\SOM_2013_admin2_Baydhaba.csv
1,SOM_2013_admin2_BeletWeyne.as,SUCCESS,784,admin2\aggregate\SOM_2013_admin2_BeletWeyne.csv
2,SOM_2013_admin2_BeletWeyne_2.as,SUCCESS,823,admin2\aggregate\SOM_2013_admin2_BeletWeyne_2.csv
3,SOM_2013_admin2_Mogadishu_IDP.as,SUCCESS,784,admin2\aggregate\SOM_2013_admin2_Mogadishu_IDP...
4,SOM_2014_11_admin1_Bari urban.as,SUCCESS,762,admin2\aggregate\SOM_2014_11_admin1_Bari urban...
5,SOM_2014_admin2_Burco_IDP.as,SUCCESS,908,admin2\aggregate\SOM_2014_admin2_Burco_IDP.csv
6,SOM_2014_admin2_Hargeysa IDP.as,SUCCESS,892,admin2\aggregate\SOM_2014_admin2_Hargeysa IDP.csv
7,Adado_SMART_survey_2022_final.as,SUCCESS,1501,admin2\individual\Adado_SMART_survey_2022_fina...
8,Burco Urban_IDPs Complete Anthro_Mortality.as,SUCCESS,1268,admin2\individual\Burco Urban_IDPs Complete An...
9,Dhobley ENA Anthro & Mortality.as,SUCCESS,1545,admin2\individual\Dhobley ENA Anthro & Mortali...


In [12]:
# ============================================================
# HELPER FUNCTIONS - DATE PROCESSING
# ============================================================

import re
from pathlib import Path

MONTHS = {
    "jan": "01", "january": "01",
    "feb": "02", "february": "02",
    "mar": "03", "march": "03",
    "apr": "04", "april": "04",
    "may": "05",
    "jun": "06", "june": "06",
    "jul": "07", "july": "07",
    "aug": "08", "august": "08",
    "sep": "09", "sept": "09", "september": "09",
    "oct": "10", "october": "10",
    "nov": "11", "november": "11",
    "dec": "12", "december": "12",
}


def get_date_order(lines):
    """
    Detect date order used by the survey.
    Returns:
        DMY
        MDY
        YMD
    """

    text = "\n".join(lines).lower()

    if "mm/dd" in text:
        return "MDY"

    if "yyyy/mm" in text:
        return "YMD"

    return "DMY"


def parse_month_year_from_date(value, order="DMY"):

    if not value:
        return None, None

    value = value.strip()

    value = value.replace("-", "/")
    value = value.replace(".", "/")

    pieces = value.split("/")

    if len(pieces) != 3:
        return None, None

    if order == "DMY":
        day, month, year = pieces

    elif order == "MDY":
        month, day, year = pieces

    else:
        year, month, day = pieces

    month = month.zfill(2)

    if len(year) == 2:
        year = "20" + year

    return month, year


def get_month_year_from_survdate_column(lines, order):

    header_found = False
    surv_idx = -1

    for line in lines:

        cells = split_cells(line)

        lower = [c.lower().strip() for c in cells]

        if not header_found:

            if "survdate" in lower:

                surv_idx = lower.index("survdate")

                header_found = True

                continue

        else:

            if surv_idx < len(cells):

                value = cells[surv_idx].strip()

                if value:

                    return parse_month_year_from_date(
                        value,
                        order
                    )

    return None, None


def parse_month_year(lines, filename):

    order = get_date_order(lines)

    month, year = get_month_year_from_survdate_column(
        lines,
        order
    )

    if month and year:
        return month, year

    name = filename.lower()

    for key, number in MONTHS.items():

        if key in name:

            month = number

            break

    else:
        month = "01"

    year_match = re.search(r"(20\d{2})", filename)

    if year_match:

        year = year_match.group(1)

    else:

        year = "2014"

    return month, year

In [ ]:
# ============================================================
# HELPER FUNCTIONS - CSV OUTPUT
# ============================================================

def clean_cell(value):

    if value is None:
        return ""

    value = str(value)

    value = value.replace("\r", " ")
    value = value.replace("\n", " ")

    return value.strip()


def row_to_csv(row):

    cleaned = []

    for cell in row:

        value = clean_cell(cell)

        if (
            "," in value
            or '"' in value
            or "\t" in value
        ):
            value = '"' + value.replace('"', '""') + '"'

        cleaned.append(value)

    return ",".join(cleaned)


def safe_write(file_path, lines):

    try:

        with open(
            file_path,
            "w",
            encoding="utf-8",
            newline=""
        ) as f:

            for line in lines:

                if line.endswith("\n"):
                    f.write(line)
                else:
                    f.write(line + "\n")

        return True

    except Exception as e:

        print(f"Write failed: {file_path.name}")

        print(e)

        return False

In [14]:
print("row_to_csv" in globals())
print("safe_write" in globals())
print("parse_month_year" in globals())

False
False
True


In [16]:
def row_to_csv(row):
    values = []

    for cell in row:
        text = "" if cell is None else str(cell)

        if '"' in text:
            text = text.replace('"', '""')

        if "," in text or '"' in text:
            text = f'"{text}"'

        values.append(text)

    return ",".join(values)


def safe_write(file_path, lines):
    with open(file_path, "w", encoding="utf-8", newline="") as f:
        for line in lines:
            if line.endswith("\n"):
                f.write(line)
            else:
                f.write(line + "\n")
    return True

In [17]:
print(row_to_csv(["a", "b", "c"]))
print("row_to_csv" in globals())
print("safe_write" in globals())

a,b,c
True
True


In [39]:
# ============================================================
# CELL 10 - BUILD CLEAN ML DATASET
# ============================================================

import csv
import pandas as pd

print("=" * 70)
print("Creating Clean ML Dataset")
print("=" * 70)

CSV_ROOT = PROJECT_DIR / "data" / "csv"
ML_ROOT = PROJECT_DIR / "data" / "ml_ready"

ML_ROOT.mkdir(parents=True, exist_ok=True)

results = []

csv_files = sorted(CSV_ROOT.rglob("*.csv"))

for file in csv_files:

    try:

        with open(file, encoding="utf-8", errors="replace") as f:
            rows = list(csv.reader(f, delimiter="\t"))

        header = None
        header_index = None

        for i, row in enumerate(rows):

            lower = [x.strip().lower() for x in row]

            if (
                "survdate" in lower
                and "cluster" in lower
                and "team" in lower
            ):
                header = row
                header_index = i
                break

        if header is None:

            results.append({
                "file": file.name,
                "status": "NO INDIVIDUAL TABLE"
            })

            continue

        ncols = len(header)

        data = []

        for row in rows[header_index + 1:]:

            if len(row) == 0:
                break

            if all(str(x).strip() == "" for x in row):
                break

            first = str(row[0]).strip().lower()

            if (
                "training_new" in first
                or "planning" in first
                or "mortality_new" in first
                or "mor_individual" in first
                or "mortality_individual" in first
                or "anthro" in first
            ):
                break

            if len(row) != ncols:
                break

            data.append(row)

        df = pd.DataFrame(data, columns=header)

        out_folder = ML_ROOT / file.parent.relative_to(CSV_ROOT)
        out_folder.mkdir(parents=True, exist_ok=True)

        out_file = out_folder / file.name

        df.to_csv(out_file, index=False)

        results.append({

            "file": file.name,
            "rows": len(df),
            "columns": len(df.columns),
            "status": "SUCCESS"

        })

    except Exception as e:

        results.append({

            "file": file.name,
            "status": "FAILED",
            "error": str(e)

        })

results = pd.DataFrame(results)

display(results)

print()
print(results.status.value_counts())
print()
print("Saved to:")
print(ML_ROOT)

Creating Clean ML Dataset


,file,rows,columns,status
0,SOM_2013_admin2_Baydhaba.csv,1,29,SUCCESS
1,SOM_2013_admin2_BeletWeyne.csv,1,29,SUCCESS
2,SOM_2013_admin2_BeletWeyne_2.csv,1,29,SUCCESS
3,SOM_2013_admin2_Mogadishu_IDP.csv,1,29,SUCCESS
4,SOM_2014_11_admin1_Bari urban.csv,1,29,SUCCESS
...,...,...,...,...
75,som_202008_lhz_eastgolis.csv,731,29,SUCCESS
76,som_202204_lhz_addun_pastoral.csv,853,29,SUCCESS
77,som_202204_lhz_bay_agropast.csv,767,29,SUCCESS
78,som_202204_lhz_nechawd_pastoral.csv,715,29,SUCCESS



status
SUCCESS    80
Name: count, dtype: int64

Saved to:
C:\Users\abdinasir\Desktop\group2 bit29b\data\ml_ready


In [40]:
# ============================================================
# CELL 11 - REMOVE EMPTY COLUMNS
# ============================================================

import pandas as pd
from pathlib import Path

ML_ROOT = PROJECT_DIR / "data" / "ml_ready"

files = list(ML_ROOT.rglob("*.csv"))

for file in files:

    df = pd.read_csv(file)

    # Remove completely empty columns
    df = df.dropna(axis=1, how="all")

    # Remove unnamed columns
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    df.to_csv(file, index=False)

print(f"Cleaned {len(files)} files.")

Cleaned 80 files.


In [41]:
# ============================================================
# CELL 12 - MERGE ALL ML-READY DATASETS
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("Merging ML-ready datasets")
print("=" * 70)

ML_ROOT = PROJECT_DIR / "data" / "ml_ready"
FINAL_ROOT = PROJECT_DIR / "data" / "final"

FINAL_ROOT.mkdir(parents=True, exist_ok=True)

csv_files = sorted(ML_ROOT.rglob("*.csv"))

print(f"Files found: {len(csv_files)}")

all_data = []

for file in csv_files:

    try:

        df = pd.read_csv(file)

        # Keep source file name
        df["SOURCE_FILE"] = file.stem

        all_data.append(df)

        print(f"✓ {file.name} ({len(df)} rows)")

    except Exception as e:

        print(f"✗ {file.name}")
        print(e)

# ------------------------------------------------------------
# Merge
# ------------------------------------------------------------

master_df = pd.concat(
    all_data,
    ignore_index=True,
    sort=False
)

master_file = FINAL_ROOT / "SMART_Master_Dataset.csv"

master_df.to_csv(
    master_file,
    index=False
)

print("\n" + "=" * 70)
print("Merge Completed")
print("=" * 70)

print(f"Rows    : {len(master_df)}")
print(f"Columns : {len(master_df.columns)}")
print(f"Saved   : {master_file}")

display(master_df.head())

Merging ML-ready datasets
Files found: 80
✓ SOM_2013_admin2_Baydhaba.csv (1 rows)
✓ SOM_2013_admin2_BeletWeyne.csv (1 rows)
✓ SOM_2013_admin2_BeletWeyne_2.csv (1 rows)
✓ SOM_2013_admin2_Mogadishu_IDP.csv (1 rows)
✓ SOM_2014_11_admin1_Bari urban.csv (1 rows)
✓ SOM_2014_admin2_Burco_IDP.csv (1 rows)
✓ SOM_2014_admin2_Hargeysa IDP.csv (1 rows)
✓ Adado_SMART_survey_2022_final.csv (671 rows)
✓ Burco Urban_IDPs Complete Anthro_Mortality.csv (502 rows)
✓ Dhobley ENA Anthro & Mortality.csv (727 rows)
✓ Elbarde_Som_Jan_2023.csv (449 rows)
✓ Hudur_Som_Jan_2023.csv (693 rows)
✓ Odweyne_rural Complete Anthro_Mortality.csv (571 rows)
✓ SOM_201806_admin2_Baydhaba.csv (1075 rows)
✓ SOM_201806_admin2_BeletWeyne rural.csv (493 rows)
✓ SOM_201806_admin2_BeletWeyne urban.csv (582 rows)
✓ SOM_201806_admin2_Berbera IDP.csv (604 rows)
✓ SOM_201806_admin2_Bossaso IDP.csv (705 rows)
✓ SOM_201806_admin2_Burco IDP.csv (635 rows)
✓ SOM_201806_admin2_Dhusamareeb IDP.csv (458 rows)
✓ SOM_201806_admin2_Doolow IDP.c

,SURVDATE,CLUSTER,TEAM,ID,HH,SOURCE_FILE,SEX,MONTHS,WEIGHT,HEIGHT,EDEMA,MUAC,WAZ,HAZ,WHZ,BIRTHDAT
0,11/29/2013,1.0,1.0,1.0,1.0,SOM_2013_admin2_Baydhaba,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2/1/14,1.0,1.0,1.0,1.0,SOM_2013_admin2_BeletWeyne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1/4/2014,1.0,1.0,1.0,1.0,SOM_2013_admin2_BeletWeyne_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12/16/2013,1.0,1.0,1.0,1.0,SOM_2013_admin2_Mogadishu_IDP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12/4/2014,1.0,1.0,1.0,1.0,SOM_2014_11_admin1_Bari urban,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
# ============================================================
# CELL 13 - DATA CLEANING & PREPROCESSING
# ============================================================

import pandas as pd

print("=" * 70)
print("Data Cleaning & Preprocessing")
print("=" * 70)

# ------------------------------------------------------------
# Make a copy
# ------------------------------------------------------------

df = master_df.copy()

# ------------------------------------------------------------
# Remove duplicate records
# ------------------------------------------------------------

duplicates = df.duplicated().sum()

print(f"Duplicate rows: {duplicates}")

df = df.drop_duplicates()

# ------------------------------------------------------------
# Convert date column
# ------------------------------------------------------------

if "SURVDATE" in df.columns:

    df["SURVDATE"] = pd.to_datetime(
        df["SURVDATE"],
        errors="coerce"
    )

# ------------------------------------------------------------
# Convert numeric columns
# ------------------------------------------------------------

numeric_columns = [

    "CLUSTER",
    "TEAM",
    "ID",
    "HH",
    "MONTHS",
    "WEIGHT",
    "HEIGHT",
    "MUAC",
    "WAZ",
    "HAZ",
    "WHZ"

]

for col in numeric_columns:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# ------------------------------------------------------------
# Missing Values
# ------------------------------------------------------------

print("\nMissing Values\n")

display(df.isnull().sum())

# ------------------------------------------------------------
# Dataset Information
# ------------------------------------------------------------

print("\nDataset Shape")

print(df.shape)

print("\nData Types")

display(df.dtypes)

print("\nFirst Five Records")

display(df.head())

# ------------------------------------------------------------
# Save Clean Dataset
# ------------------------------------------------------------

clean_file = FINAL_ROOT / "SMART_Clean_Dataset.csv"

df.to_csv(
    clean_file,
    index=False
)

print("\nSaved to:")

print(clean_file)

Data Cleaning & Preprocessing
Duplicate rows: 9

Missing Values



SURVDATE        3300
CLUSTER         1451
TEAM             556
ID             28440
HH              1123
SOURCE_FILE        0
SEX              135
MONTHS           137
WEIGHT           140
HEIGHT           141
EDEMA            115
MUAC              38
WAZ              193
HAZ              152
WHZ              207
BIRTHDAT       42368
dtype: int64


Dataset Shape
(43870, 16)

Data Types


SURVDATE       datetime64[ns]
CLUSTER               float64
TEAM                  float64
ID                    float64
HH                    float64
SOURCE_FILE            object
SEX                    object
MONTHS                float64
WEIGHT                float64
HEIGHT                float64
EDEMA                  object
MUAC                  float64
WAZ                   float64
HAZ                   float64
WHZ                   float64
BIRTHDAT               object
dtype: object


First Five Records


,SURVDATE,CLUSTER,TEAM,ID,HH,SOURCE_FILE,SEX,MONTHS,WEIGHT,HEIGHT,EDEMA,MUAC,WAZ,HAZ,WHZ,BIRTHDAT
0,2013-11-29,1.0,1.0,1.0,1.0,SOM_2013_admin2_Baydhaba,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaT,1.0,1.0,1.0,1.0,SOM_2013_admin2_BeletWeyne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-01-04,1.0,1.0,1.0,1.0,SOM_2013_admin2_BeletWeyne_2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-12-16,1.0,1.0,1.0,1.0,SOM_2013_admin2_Mogadishu_IDP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-12-04,1.0,1.0,1.0,1.0,SOM_2014_11_admin1_Bari urban,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved to:
C:\Users\abdinasir\Desktop\group2 bit29b\data\final\SMART_Clean_Dataset.csv


In [43]:
# ============================================================
# CELL 14 - EXTRACT MORTALITY TABLES
# ============================================================

import csv
import pandas as pd

print("=" * 70)
print("Extracting Mortality Tables")
print("=" * 70)

CSV_ROOT = PROJECT_DIR / "data" / "csv"
MORTALITY_ROOT = PROJECT_DIR / "data" / "mortality"

MORTALITY_ROOT.mkdir(parents=True, exist_ok=True)

results = []

csv_files = sorted(CSV_ROOT.rglob("*.csv"))

for file in csv_files:

    try:

        with open(file, encoding="utf-8", errors="replace") as f:
            lines = f.readlines()

        start = None

        for i, line in enumerate(lines):

            if "mortality_new" in line.lower():

                start = i + 1
                break

        if start is None:

            results.append({
                "file": file.name,
                "status": "NO MORTALITY TABLE"
            })

            continue

        mortality_lines = []

        for line in lines[start:]:

            if not line.strip():
                break

            mortality_lines.append(line)

        out_folder = MORTALITY_ROOT / file.parent.relative_to(CSV_ROOT)
        out_folder.mkdir(parents=True, exist_ok=True)

        out_file = out_folder / file.name

        with open(out_file, "w", encoding="utf-8") as out:

            out.writelines(mortality_lines)

        results.append({

            "file": file.name,
            "rows": len(mortality_lines),
            "status": "SUCCESS"

        })

    except Exception as e:

        results.append({

            "file": file.name,
            "status": "FAILED",
            "error": str(e)

        })

results = pd.DataFrame(results)

display(results.head(20))

print()
print(results.status.value_counts())

print()
print("Saved to:")
print(MORTALITY_ROOT)

Extracting Mortality Tables


,file,rows,status
0,SOM_2013_admin2_Baydhaba.csv,630,SUCCESS
1,SOM_2013_admin2_BeletWeyne.csv,512,SUCCESS
2,SOM_2013_admin2_BeletWeyne_2.csv,552,SUCCESS
3,SOM_2013_admin2_Mogadishu_IDP.csv,513,SUCCESS
4,SOM_2014_11_admin1_Bari urban.csv,661,SUCCESS
5,SOM_2014_admin2_Burco_IDP.csv,637,SUCCESS
6,SOM_2014_admin2_Hargeysa IDP.csv,621,SUCCESS
7,Adado_SMART_survey_2022_final.csv,532,SUCCESS
8,Burco Urban_IDPs Complete Anthro_Mortality.csv,496,SUCCESS
9,Dhobley ENA Anthro & Mortality.csv,548,SUCCESS



status
SUCCESS    80
Name: count, dtype: int64

Saved to:
C:\Users\abdinasir\Desktop\group2 bit29b\data\mortality


In [44]:
# ============================================================
# CELL 15 - PREVIEW MORTALITY DATA
# ============================================================

from pathlib import Path

MORTALITY_ROOT = PROJECT_DIR / "data" / "mortality"

files = sorted(MORTALITY_ROOT.rglob("*.csv"))

print(f"Mortality files: {len(files)}")

print("\nFirst file:")
print(files[0])

print("\n" + "=" * 70)

with open(files[0], encoding="utf-8", errors="replace") as f:

    for i in range(30):

        line = f.readline()

        if not line:
            break

        print(f"{i+1:02d}: {line.rstrip()}")

Mortality files: 80

First file:
C:\Users\abdinasir\Desktop\group2 bit29b\data\mortality\admin2\aggregate\SOM_2013_admin2_Baydhaba.csv

01: 22	22	1	1	0	0	0	2	0	0	0
02: 21,29,2013,0,0,0,0,0,0,0,0,
03: 20,29,2013,0,0,0,0,0,0,0,0,
04: 19,29,2013,2,0,0,0,0,0,0,0,
05: 18,29,2013,0,0,0,0,0,0,0,0,
06: 17,29,2013,1,0,0,0,0,0,0,0,
07: 16,29,2013,2,0,0,0,0,0,0,0,
08: 15,29,2013,0,1,0,0,0,0,0,0,
09: 14,29,2013,2,0,0,0,0,1,0,0,
10: 13,29,2013,1,0,0,0,0,0,0,0,
11: 12,29,2013,2,1,0,1,0,0,0,0,
12: 11,29,2013,2,0,0,0,0,0,0,0,
13: 10,29,2013,0,0,0,0,0,0,0,0,
14: 9,29,2013,2,0,0,0,0,0,0,0,
15: 8,29,2013,2,2,0,0,0,0,0,0,
16: 7,29,2013,1,0,0,0,0,0,2,1,
17: 5,29,2013,2,0,0,0,0,0,0,0,
18: 6,29,2013,0,0,0,0,0,0,1,0,
19: 4,29,2013,2,0,0,0,0,0,0,0,
20: 3,29,2013,1,0,0,1,0,0,1,1,
21: 2,29,2013,2,0,0,0,0,0,0,0,
22: 1,29,2013,2,0,0,0,0,0,0,0,
23: 22,29,2013,1,0,0,0,0,0,0,0,
24: 21,29,2013,0,0,0,0,0,0,0,0,
25: 20,29,2013,2,0,0,0,0,0,0,0,
26: 19,29,2013,3,0,0,0,0,0,0,0,
27: 18,29,2013,0,0,0,0,0,0,0,0,
28: 17,29,201

In [45]:
# ============================================================
# CELL 16 - INSPECT MORTALITY RECORDS
# ============================================================

from pathlib import Path
import csv

MORTALITY_ROOT = PROJECT_DIR / "data" / "mortality"

file = sorted(MORTALITY_ROOT.rglob("*.csv"))[0]

print(file)

rows = []

with open(file, encoding="utf-8", errors="replace") as f:

    reader = csv.reader(f)

    for row in reader:

        if len(row) > 0:

            rows.append(row)

print("\nFirst 10 rows:\n")

for r in rows[:10]:

    print(r)

print("\nNumber of columns:")

print(max(len(r) for r in rows))

C:\Users\abdinasir\Desktop\group2 bit29b\data\mortality\admin2\aggregate\SOM_2013_admin2_Baydhaba.csv

First 10 rows:

['22\t22\t1\t1\t0\t0\t0\t2\t0\t0\t0\t']
['21', '29', '2013', '0', '0', '0', '0', '0', '0', '0', '0', '']
['20', '29', '2013', '0', '0', '0', '0', '0', '0', '0', '0', '']
['19', '29', '2013', '2', '0', '0', '0', '0', '0', '0', '0', '']
['18', '29', '2013', '0', '0', '0', '0', '0', '0', '0', '0', '']
['17', '29', '2013', '1', '0', '0', '0', '0', '0', '0', '0', '']
['16', '29', '2013', '2', '0', '0', '0', '0', '0', '0', '0', '']
['15', '29', '2013', '0', '1', '0', '0', '0', '0', '0', '0', '']
['14', '29', '2013', '2', '0', '0', '0', '0', '1', '0', '0', '']
['13', '29', '2013', '1', '0', '0', '0', '0', '0', '0', '0', '']

Number of columns:
12


In [46]:
# ============================================================
# CELL 17 - CHECK FIRST MORTALITY ROW
# ============================================================

from pathlib import Path

file = sorted((PROJECT_DIR / "data" / "mortality").rglob("*.csv"))[0]

with open(file, encoding="utf-8", errors="replace") as f:

    first = f.readline().strip()

print(first)

print()

print(first.split("\t"))

print()

print("Columns:", len(first.split("\t")))

22	22	1	1	0	0	0	2	0	0	0

['22', '22', '1', '1', '0', '0', '0', '2', '0', '0', '0']

Columns: 11


In [47]:
# ============================================================
# CELL 18 - NORMALIZE MORTALITY CSV FILES
# ============================================================

import csv
from pathlib import Path
import pandas as pd

MORTALITY_ROOT = PROJECT_DIR / "data" / "mortality"
NORMALIZED_ROOT = PROJECT_DIR / "data" / "mortality_clean"

NORMALIZED_ROOT.mkdir(parents=True, exist_ok=True)

results = []

for file in sorted(MORTALITY_ROOT.rglob("*.csv")):

    out_folder = NORMALIZED_ROOT / file.parent.relative_to(MORTALITY_ROOT)
    out_folder.mkdir(parents=True, exist_ok=True)

    out_file = out_folder / file.name

    rows = []

    with open(file, encoding="utf-8", errors="replace") as f:

        for i, line in enumerate(f):

            line = line.strip()

            if not line:
                continue

            # First row is TAB separated
            if i == 0:
                row = line.split("\t")

            # Remaining rows are comma separated
            else:
                row = line.split(",")

            # Remove empty values caused by trailing comma
            while len(row) > 0 and row[-1] == "":
                row.pop()

            rows.append(row)

    max_cols = max(len(r) for r in rows)

    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    with open(out_file, "w", newline="", encoding="utf-8") as f:

        writer = csv.writer(f)
        writer.writerows(rows)

    results.append({

        "file": file.name,
        "rows": len(rows),
        "columns": max_cols,
        "status": "SUCCESS"

    })

results = pd.DataFrame(results)

display(results.head())

print()
print(results.status.value_counts())

print()
print("Saved to:")
print(NORMALIZED_ROOT)

,file,rows,columns,status
0,SOM_2013_admin2_Baydhaba.csv,630,11,SUCCESS
1,SOM_2013_admin2_BeletWeyne.csv,512,11,SUCCESS
2,SOM_2013_admin2_BeletWeyne_2.csv,552,11,SUCCESS
3,SOM_2013_admin2_Mogadishu_IDP.csv,513,11,SUCCESS
4,SOM_2014_11_admin1_Bari urban.csv,661,11,SUCCESS



status
SUCCESS    80
Name: count, dtype: int64

Saved to:
C:\Users\abdinasir\Desktop\group2 bit29b\data\mortality_clean


In [48]:
# ============================================================
# CELL 19 - VERIFY NORMALIZED MORTALITY FILES
# ============================================================

from pathlib import Path
import pandas as pd

MORTALITY_CLEAN = PROJECT_DIR / "data" / "mortality_clean"

files = sorted(MORTALITY_CLEAN.rglob("*.csv"))

print(f"Files found: {len(files)}")

df = pd.read_csv(files[0], header=None)

print("\nShape:")
print(df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nMissing values:")
print(df.isna().sum())

Files found: 80

Shape:
(630, 11)

First five rows:


,0,1,2,3,4,5,6,7,8,9,10
0,22,22.0,1.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,21,29.0,2013.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,20,29.0,2013.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,19,29.0,2013.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,18,29.0,2013.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Missing values:
0     0
1     2
2     2
3     2
4     2
5     2
6     2
7     2
8     2
9     2
10    2
dtype: int64


In [49]:
# ============================================================
# CELL 20 - MERGE MORTALITY DATASETS
# ============================================================

import pandas as pd
from pathlib import Path

MORTALITY_CLEAN = PROJECT_DIR / "data" / "mortality_clean"
FINAL_ROOT = PROJECT_DIR / "data" / "final"

FINAL_ROOT.mkdir(parents=True, exist_ok=True)

files = sorted(MORTALITY_CLEAN.rglob("*.csv"))

all_data = []

for file in files:

    df = pd.read_csv(file, header=None)

    # Remove completely empty rows
    df = df.dropna(how="all")

    # Remove rows with missing values
    df = df.dropna()

    # Keep source file
    df["SOURCE_FILE"] = file.stem

    all_data.append(df)

master = pd.concat(
    all_data,
    ignore_index=True
)

master.to_csv(
    FINAL_ROOT / "SMART_Mortality_Master.csv",
    index=False
)

print("=" * 60)
print("Merge Completed")
print("=" * 60)

print("Rows   :", len(master))
print("Columns:", len(master.columns))

display(master.head())

C:\Users\abdinasir\AppData\Local\Temp\ipykernel_21116\3013225692.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master = pd.concat(


Merge Completed
Rows   : 8643
Columns: 143


,0,1,2,3,4,5,6,7,8,9,...,132,133,134,135,136,137,138,139,140,141
0,22,22.0,1.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,21,29.0,2013.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20,29.0,2013.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19,29.0,2013.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,18,29.0,2013.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [50]:
# ============================================================
# CELL 21 - REMOVE UNUSED FILES
# ============================================================

from pathlib import Path
import shutil

DATA = PROJECT_DIR / "data"

items = [
    DATA / "final" / "SMART_Master_Dataset.csv",
    DATA / "final" / "SMART_Mortality_Master.csv",
    DATA / "mortality_clean",
]

for item in items:

    if item.exists():

        if item.is_dir():
            shutil.rmtree(item)
            print(f"Removed folder: {item}")

        else:
            item.unlink()
            print(f"Removed file: {item}")

    else:
        print(f"Not found: {item}")

print("\nCleanup completed.")

Removed file: C:\Users\abdinasir\Desktop\group2 bit29b\data\final\SMART_Master_Dataset.csv
Removed file: C:\Users\abdinasir\Desktop\group2 bit29b\data\final\SMART_Mortality_Master.csv
Removed folder: C:\Users\abdinasir\Desktop\group2 bit29b\data\mortality_clean

Cleanup completed.


In [58]:
# ============================================================
# CELL 22 - CREATE mortality_household.csv
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\abdinasir\Desktop\group2 bit29b")

INDIVIDUAL_DIR = PROJECT_DIR / "data" / "mortality" / "admin2" / "individual"
FINAL_DIR = PROJECT_DIR / "data" / "final"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

all_data = []

for file in sorted(INDIVIDUAL_DIR.glob("*.csv")):

    with open(file, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    header = None
    start = None

    for i, line in enumerate(lines):

        if "Date\tCluster\tTeam\tHH" in line:

            header = line.strip().split("\t")
            start = i + 1
            break

    if header is None:
        continue

    rows = []

    for line in lines[start:]:

        line = line.strip()

        if line:
            rows.append(line.split(","))

    # expand header if data has more columns
    max_cols = max(len(r) for r in rows)

    if len(header) < max_cols:

        for i in range(len(header), max_cols):
            header.append(f"extra_{i}")

    # adjust rows
    rows = [
        r + [""] * (len(header) - len(r))
        for r in rows
    ]

    df = pd.DataFrame(rows, columns=header)

    df["SOURCE_FILE"] = file.stem

    all_data.append(df)


mortality_household = pd.concat(
    all_data,
    ignore_index=True,
    sort=False
)

output = FINAL_DIR / "mortality_household.csv"

mortality_household.to_csv(output, index=False)

print("="*60)
print("mortality_household created")
print("="*60)

print("Rows:", len(mortality_household))
print("Columns:", len(mortality_household.columns))
print("Saved:", output)

mortality_household created
Rows: 26392
Columns: 165
Saved: C:\Users\abdinasir\Desktop\group2 bit29b\data\final\mortality_household.csv


In [59]:
# ============================================================
# CELL 23 - CREATE mortality_legacy.csv
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\abdinasir\Desktop\group2 bit29b")

AGG_DIR = PROJECT_DIR / "data" / "mortality" / "admin2" / "aggregate"
FINAL_DIR = PROJECT_DIR / "data" / "final"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

all_data = []

for file in sorted(AGG_DIR.glob("*.csv")):

    rows = []

    with open(file, encoding="utf-8", errors="replace") as f:

        for line in f:

            line = line.strip()

            if line:
                rows.append(line.replace("\t", ",").split(","))

    # remove empty trailing columns
    rows = [
        [x for x in row if x != ""]
        for row in rows
    ]

    if rows:

        df = pd.DataFrame(rows)

        df["SOURCE_FILE"] = file.stem

        all_data.append(df)


legacy = pd.concat(
    all_data,
    ignore_index=True,
    sort=False
)

output = FINAL_DIR / "mortality_legacy.csv"

legacy.to_csv(
    output,
    index=False
)

print("="*60)
print("mortality_legacy created")
print("="*60)

print("Rows:", len(legacy))
print("Columns:", len(legacy.columns))
print("Saved:", output)

mortality_legacy created
Rows: 4126
Columns: 165
Saved: C:\Users\abdinasir\Desktop\group2 bit29b\data\final\mortality_legacy.csv
